# ARM97 Multiple Model Outputs vs IOP Observation

This notebook compares manually configured ARM97 SCM model output NetCDF files against the same ARM97 IOP observation file. Observation time is interpreted as `bdate + tsec`, and each model is aligned to the observation on its own model time axis before statistics are calculated.

Edit `MODEL_SPECS` in the setup cell to choose model outputs and display labels. Paths can be absolute or relative to the repository root.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta
import os
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OBSERVATION = Path(
    os.environ.get(
        "ARM97_OBSERVATION_FILE",
        str(ROOT / "outputs/arm97_iop_observation.nc"),
    )
).expanduser().resolve()


def model_path(path_like):
    path = Path(path_like).expanduser()
    if not path.is_absolute():
        path = ROOT / path
    return path.resolve()


# Manually configure model-ready output files here. Do not use the observation
# window files as model inputs. Labels are used in legends, statistics tables,
# and PDF reports.
MODEL_SPECS = [
    (
        "baseline",
        model_path("cases/run_e3sm_scm_ARM97/arm97_model_ready.nc"),
    ),
    (
        "implicit",
        model_path("cases/run_e3sm_scm_ARM97_implicit_stress/arm97_implicit_stress_model_ready.nc"),
    ),
]

OUT_DIR = ROOT / "notebook_outputs" / "arm97_multiple_models_vs_observation"
FIG_DIR = OUT_DIR / "figures"

assert ROOT.exists(), ROOT
assert OBSERVATION.exists(), OBSERVATION
assert MODEL_SPECS, "MODEL_SPECS is empty. Add at least one (label, path) entry."
for label, file in MODEL_SPECS:
    assert file.exists(), file

print("repo root:", ROOT)
print("observation:", OBSERVATION)
print("models:")
for label, file in MODEL_SPECS:
    print(f"  {label}: {file}")
print("output dir:", OUT_DIR)

## Variable Mapping

Observation variables are converted to model units before statistics are calculated. Expressions such as `srfswdn-srfswup` are evaluated from observation variables before interpolation.

In [ ]:
@dataclass(frozen=True)
class VarSpec:
    model: str
    obs: str
    units: str
    scale_obs: float = 1.0
    obs_offset: float = 0.0
    description: str = ""


SURFACE_VARS = [
    VarSpec("TREFHT", "Tsair", "K", description="2 m air temperature"),
    VarSpec("TS", "Tg", "K", description="surface/ground temperature"),
    VarSpec("TMQ", "prew", "kg/m2", scale_obs=10.0, description="precipitable water"),
    VarSpec("CLDTOT", "totcld", "1", scale_obs=0.01, description="total cloud fraction"),
    VarSpec("CLDLOW", "lowcld", "1", scale_obs=0.01, description="low cloud fraction"),
    VarSpec("CLDMED", "midcld", "1", scale_obs=0.01, description="mid-level cloud fraction"),
    VarSpec("CLDHGH", "hghcld", "1", scale_obs=0.01, description="high cloud fraction"),
    VarSpec("PS", "Ps", "Pa", description="surface pressure"),
    VarSpec("LHFLX", "lhflx", "W/m2", description="latent heat flux"),
    VarSpec("SHFLX", "shflx", "W/m2", description="sensible heat flux"),
    VarSpec("FSNS", "srfswdn-srfswup", "W/m2", description="surface net shortwave flux"),
    VarSpec("FLNS", "srflwup-srflwdn", "W/m2", description="surface net longwave flux"),
    VarSpec("FSDS", "srfswdn", "W/m2", description="surface downwelling shortwave flux"),
    VarSpec("FLDS", "srflwdn", "W/m2", description="surface downwelling longwave flux"),
    VarSpec("FLUT", "TOA_LWup", "W/m2", description="TOA upwelling longwave flux"),
    VarSpec("U10", "windsrf", "m/s", description="10 m wind speed"),
    VarSpec("PRECT", "Prec", "m/s", scale_obs=0.001, description="total precipitation rate"),
]

pd.DataFrame([vars(v) for v in SURFACE_VARS])

## Load And Align Surface Data

Each model file keeps its own time axis. Observation values are interpolated onto that model time axis using absolute datetimes derived from `bdate + tsec`.

In [ ]:
def as_series(var):
    data = np.ma.asarray(var[:], dtype=np.float64)
    if data.ndim == 1:
        return data
    axes = tuple(range(1, data.ndim))
    return np.ma.mean(data, axis=axes)


def obs_series(ds, expression: str):
    expression = expression.replace(" ", "")
    if "-" in expression:
        left, right = expression.split("-", 1)
        return as_series(ds.variables[left]) - as_series(ds.variables[right])
    return as_series(ds.variables[expression])


def filled(arr):
    return np.asarray(np.ma.asarray(arr, dtype=np.float64).filled(np.nan), dtype=np.float64)


def load_model_time_axis(ds):
    time = ds.variables["time"]
    days = np.asarray(time[:], dtype=np.float64)
    units = getattr(time, "units", "")
    if "since" not in units:
        raise ValueError(
            f"{ds.filepath()} is not a model-ready output file: time.units={units!r}. "
            "Use files such as arm97_model_ready.nc or *_model_ready.nc in MODEL_SPECS."
        )
    dates = np.array(
        num2date(days, units, getattr(time, "calendar", "standard"), only_use_cftime_datetimes=False),
        dtype=object,
    )
    return days, dates


def parse_bdate(ds):
    value = int(np.asarray(ds.variables["bdate"][...]).item())
    text = str(value)
    if len(text) == 8:
        return datetime(int(text[:4]), int(text[4:6]), int(text[6:8]))
    if len(text) == 6:
        year = int(text[:2])
        year += 1900 if year >= 70 else 2000
        return datetime(year, int(text[2:4]), int(text[4:6]))
    raise ValueError(f"unsupported bdate value: {value}")


def load_obs_dates(ds):
    base = parse_bdate(ds)
    tsec = np.asarray(ds.variables["tsec"][:], dtype=np.float64)
    return np.array([base + timedelta(seconds=float(x)) for x in tsec], dtype=object)


def obs_relative_days_from_model_origin(obs_dates, origin):
    return np.asarray([(d - origin).total_seconds() / 86400.0 for d in obs_dates], dtype=np.float64)


def interpolate_obs(obs_days, obs_values, target_days):
    finite = np.isfinite(obs_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    return np.interp(target_days, obs_days[finite], obs_values[finite], left=np.nan, right=np.nan)


def stats(model_values, obs_values):
    finite = np.isfinite(model_values) & np.isfinite(obs_values)
    diff = model_values[finite] - obs_values[finite]
    if diff.size == 0:
        return dict(n=0, mean_obs=np.nan, mean_model=np.nan, bias=np.nan, mae=np.nan, rmse=np.nan, max_abs=np.nan)
    return dict(
        n=int(diff.size),
        mean_obs=float(np.mean(obs_values[finite])),
        mean_model=float(np.mean(model_values[finite])),
        bias=float(np.mean(diff)),
        mae=float(np.mean(np.abs(diff))),
        rmse=float(np.sqrt(np.mean(diff * diff))),
        max_abs=float(np.max(np.abs(diff))),
    )


SURFACE_DATA = {}
summary_rows = []
with Dataset(OBSERVATION) as obs:
    obs_dates = load_obs_dates(obs)
    for label, model_file in MODEL_SPECS:
        with Dataset(model_file) as model:
            model_days, model_dates = load_model_time_axis(model)
            origin = model_dates[0] - timedelta(days=float(model_days[0]))
            obs_days = obs_relative_days_from_model_origin(obs_dates, origin)

            for spec in SURFACE_VARS:
                obs_names = [name.strip() for name in spec.obs.replace("-", ",").split(",")]
                if spec.model not in model.variables or any(name not in obs.variables for name in obs_names):
                    continue
                model_values = filled(as_series(model.variables[spec.model]))
                obs_native = filled(obs_series(obs, spec.obs)) * spec.scale_obs + spec.obs_offset
                obs_at_model = interpolate_obs(obs_days, obs_native, model_days)
                model_stats = stats(model_values, obs_at_model)
                item = dict(
                    spec=spec,
                    label=label,
                    model_file=model_file,
                    model_dates=model_dates,
                    obs_dates=obs_dates,
                    model_days=model_days,
                    obs_days=obs_days,
                    model_values=model_values,
                    obs_native=obs_native,
                    obs_at_model=obs_at_model,
                    diff=model_values - obs_at_model,
                    stats=model_stats,
                )
                SURFACE_DATA.setdefault(spec.model, {})[label] = item
                summary_rows.append(
                    {
                        "variable": spec.model,
                        "model": label,
                        "observation": spec.obs,
                        "description": spec.description,
                        "units": spec.units,
                        **model_stats,
                    }
                )

summary = pd.DataFrame(summary_rows).sort_values(["variable", "rmse", "model"])
print(f"Loaded {len(SURFACE_DATA)} surface variables across {len(MODEL_SPECS)} model file(s).")
summary

## Interactive Surface Comparison

Use the dropdown to switch variables. The top panel overlays all model outputs with observation; the bottom panel shows `model - observation` for each model on its native time axis.

In [ ]:
MODEL_COLORS = [
    "#1261A6",  # blue
    "#D97706",  # amber
    "#7C3AED",  # violet
    "#059669",  # green
    "#DC2626",  # red
    "#0891B2",  # cyan
    "#BE185D",  # rose
    "#4B5563",  # gray
]


def display_label(label):
    text = str(label).replace("_", " ").replace("-", " ").strip()
    lowered = text.lower()
    if "continuous" in lowered:
        return "continuous baseline"
    if "stitched" in lowered:
        return "stitched baseline"
    if "model ready" in lowered:
        return "continuous baseline"
    if "baseline" in lowered:
        return "baseline"
    return text


def title_for(name, items):
    spec = next(iter(items.values()))["spec"]
    rmse_parts = []
    rmse_values = []
    for label, d in items.items():
        rmse = d["stats"]["rmse"]
        rmse_values.append(rmse)
        rmse_parts.append(f"{display_label(label)} RMSE={rmse:.3g}")
    if len(rmse_values) >= 2 and np.isfinite(rmse_values[0]) and np.isfinite(rmse_values[1]):
        rmse_parts.append(f"delta={rmse_values[1] - rmse_values[0]:.3g}")
    return f"<b>{name}: {spec.description}</b><br><sup>Units: {spec.units}; " + ", ".join(rmse_parts) + "</sup>"


fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.68, 0.32],
)

buttons = []
trace_groups = []
for variable_index, (name, items) in enumerate(SURFACE_DATA.items()):
    visible = variable_index == 0
    start = len(fig.data)
    spec = next(iter(items.values()))["spec"]

    for model_index, (label, d) in enumerate(items.items()):
        color = MODEL_COLORS[model_index % len(MODEL_COLORS)]
        shown_label = display_label(label)
        fig.add_trace(
            go.Scatter(
                x=d["model_dates"],
                y=d["model_values"],
                mode="lines",
                name=shown_label,
                legendgroup=shown_label,
                legendrank=20 + model_index,
                line=dict(color=color, width=2.1),
                opacity=0.90,
                visible=visible,
                hovertemplate=f"%{{x}}<br>{shown_label}=%{{y:.4g}}<extra></extra>",
            ),
            row=1,
            col=1,
        )

    first = next(iter(items.values()))
    # Add observation after model traces so it is drawn above all model lines.
    fig.add_trace(
        go.Scatter(
            x=first["obs_dates"],
            y=first["obs_native"],
            mode="lines+markers",
            name="observation",
            legendgroup="observation",
            legendrank=1,
            line=dict(color="#4A4A4A", width=2.6),
            marker=dict(color="#4A4A4A", size=3, opacity=0.80),
            visible=visible,
            hovertemplate="%{x}<br>obs=%{y:.4g}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    for model_index, (label, d) in enumerate(items.items()):
        color = MODEL_COLORS[model_index % len(MODEL_COLORS)]
        shown_label = display_label(label)
        fig.add_trace(
            go.Scatter(
                x=d["model_dates"],
                y=d["diff"],
                mode="lines",
                name=f"{shown_label} - observation",
                legendgroup=shown_label,
                showlegend=False,
                line=dict(color=color, width=1.4),
                opacity=0.82,
                visible=visible,
                hovertemplate=f"%{{x}}<br>{shown_label} - obs=%{{y:.4g}}<extra></extra>",
            ),
            row=2,
            col=1,
        )

    stop = len(fig.data)
    trace_groups.append((start, stop))
    buttons.append(
        dict(
            label=name,
            method="update",
            args=[
                {"visible": []},
                {
                    "title.text": title_for(name, items),
                    "yaxis.title.text": f"{name} ({spec.units})",
                    "yaxis2.title.text": f"model - obs ({spec.units})",
                },
            ],
        )
    )

# Plotly update buttons need masks with final trace count.
for button, (start, stop) in zip(buttons, trace_groups):
    mask = [False] * len(fig.data)
    for idx in range(start, stop):
        mask[idx] = True
    button["args"][0]["visible"] = mask

first_name = next(iter(SURFACE_DATA))
first_items = SURFACE_DATA[first_name]
first_spec = next(iter(first_items.values()))["spec"]
fig.update_layout(
    title=dict(text=title_for(first_name, first_items), x=0.01, xanchor="left", font=dict(size=22)),
    template="plotly_white",
    height=740,
    width=1120,
    hovermode="x unified",
    margin=dict(l=80, r=40, t=110, b=70),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.58,
        xanchor="center",
        x=0.50,
        bgcolor="rgba(255,255,255,0.65)",
        borderwidth=0,
    ),
    updatemenus=[
        dict(type="dropdown", x=1.0, y=1.12, xanchor="right", yanchor="top", buttons=buttons, showactive=True)
    ],
)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(title="Time", row=2, col=1)
fig.update_yaxes(title=f"{first_name} ({first_spec.units})", row=1, col=1)
fig.update_yaxes(
    title=f"model - obs ({first_spec.units})",
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor="black",
    row=2,
    col=1,
)
fig.show()


## Profile Variables

For profile variables, model hybrid levels are converted to pressure with `hyam * P0 + hybm * PS(time)`, then interpolated to the selected observation pressure level. The controls require `ipywidgets`; if widgets are unavailable, the data dictionaries and summary tables are still available for static plotting.

In [ ]:
@dataclass(frozen=True)
class ProfileSpec:
    model: str
    obs: str
    units: str
    description: str = ""


PROFILE_VARS = [
    ProfileSpec("T", "T", "K", "temperature"),
    ProfileSpec("Q", "q", "kg/kg", "specific humidity"),
    ProfileSpec("U", "u", "m/s", "zonal wind"),
    ProfileSpec("V", "v", "m/s", "meridional wind"),
    ProfileSpec("OMEGA", "omega", "Pa/s", "pressure vertical velocity"),
    ProfileSpec("RELHUM", "rh", "%", "relative humidity"),
]


def interp_model_matrix_to_pressure(values, pressure, target_pressure):
    values = np.asarray(values, dtype=np.float64)
    pressure = np.asarray(pressure, dtype=np.float64)
    idx = np.sum(pressure < target_pressure, axis=1)
    valid = (idx > 0) & (idx < pressure.shape[1])
    out = np.full(values.shape[0], np.nan, dtype=np.float64)
    if not valid.any():
        return out

    rows = np.arange(values.shape[0])[valid]
    upper = idx[valid]
    lower = upper - 1
    p0 = pressure[rows, lower]
    p1 = pressure[rows, upper]
    v0 = values[rows, lower]
    v1 = values[rows, upper]
    ok = np.isfinite(p0) & np.isfinite(p1) & np.isfinite(v0) & np.isfinite(v1) & (p1 != p0)
    interp = np.full(rows.shape[0], np.nan, dtype=np.float64)
    interp[ok] = v0[ok] + (target_pressure - p0[ok]) * (v1[ok] - v0[ok]) / (p1[ok] - p0[ok])
    out[rows] = interp
    return out


def model_pressure(ds):
    p0 = float(np.asarray(ds.variables["P0"][...]))
    hyam = np.asarray(ds.variables["hyam"][:], dtype=np.float64)
    hybm = np.asarray(ds.variables["hybm"][:], dtype=np.float64)
    ps = np.asarray(ds.variables["PS"][:], dtype=np.float64).squeeze()
    return hyam[None, :] * p0 + hybm[None, :] * ps[:, None]


def load_profile_data():
    profile = {}
    rows = []
    with Dataset(OBSERVATION) as obs:
        obs_dates = load_obs_dates(obs)
        obs_levels_pa = np.asarray(obs.variables["lev"][:], dtype=np.float64)

        for label, model_file in MODEL_SPECS:
            with Dataset(model_file) as model:
                required = {"P0", "hyam", "hybm", "PS"}
                if not required.issubset(model.variables):
                    continue
                model_days, model_dates = load_model_time_axis(model)
                origin = model_dates[0] - timedelta(days=float(model_days[0]))
                obs_days = obs_relative_days_from_model_origin(obs_dates, origin)
                pressure = model_pressure(model)

                for spec in PROFILE_VARS:
                    if spec.model not in model.variables or spec.obs not in obs.variables:
                        continue
                    model_values = np.asarray(model.variables[spec.model][:], dtype=np.float64).squeeze()
                    obs_values = np.asarray(obs.variables[spec.obs][:], dtype=np.float64).squeeze()
                    if obs_values.ndim > 2:
                        obs_values = np.nanmean(obs_values, axis=tuple(range(2, obs_values.ndim)))

                    variable = profile.setdefault(spec.model, {"spec": spec, "obs_levels_pa": obs_levels_pa, "models": {}})
                    model_levels = {}
                    for level_index, target_pressure in enumerate(obs_levels_pa):
                        model_at_level = interp_model_matrix_to_pressure(model_values, pressure, target_pressure)
                        obs_native = obs_values[:, level_index]
                        obs_at_model = interpolate_obs(obs_days, obs_native, model_days)
                        level_stats = stats(model_at_level, obs_at_model)
                        model_levels[float(target_pressure)] = dict(
                            model_at_level=model_at_level,
                            obs_native=obs_native,
                            obs_at_model=obs_at_model,
                            diff=model_at_level - obs_at_model,
                            stats=level_stats,
                        )
                        rows.append(
                            {
                                "variable": spec.model,
                                "model": label,
                                "level_hpa": float(target_pressure) / 100.0,
                                "units": spec.units,
                                **level_stats,
                            }
                        )
                    variable["models"][label] = dict(
                        model_file=model_file,
                        model_dates=model_dates,
                        obs_dates=obs_dates,
                        model_days=model_days,
                        levels=model_levels,
                    )
    return profile, pd.DataFrame(rows)


PROFILE_DATA, profile_summary = load_profile_data()
print(f"Loaded {len(PROFILE_DATA)} profile variables across {len(MODEL_SPECS)} model file(s).")
profile_summary.head()

In [ ]:
try:
    import ipywidgets as widgets
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt
    from IPython.display import display

    profile_names = list(PROFILE_DATA)
    assert profile_names, "No profile variables were loaded."
    pressure_options = [
        (f"{p / 100:.0f} hPa", float(p))
        for p in next(iter(PROFILE_DATA.values()))["obs_levels_pa"]
        if float(p) < 96500.0
    ]
    pressure_values = [value for _, value in pressure_options]
    default_pressure = min(pressure_values, key=lambda p: abs(p - 50000.0))

    profile_var = widgets.Dropdown(
        options=[(f"{name}: {PROFILE_DATA[name]['spec'].description}", name) for name in profile_names],
        value="T" if "T" in profile_names else profile_names[0],
        description="variable",
        layout=widgets.Layout(width="430px"),
    )
    pressure_level = widgets.SelectionSlider(
        options=pressure_options,
        value=default_pressure,
        description="level",
        continuous_update=False,
        readout=True,
        layout=widgets.Layout(width="620px"),
        style={"description_width": "50px"},
    )

    def draw_profile(name, level_pa):
        level_pa = float(level_pa)
        variable = PROFILE_DATA[name]
        spec = variable["spec"]
        fig, axes = plt.subplots(
            2,
            1,
            figsize=(12, 7),
            sharex=True,
            gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08},
        )

        first_model = next(iter(variable["models"].values()))
        first_level = first_model["levels"][level_pa]
        axes[0].plot(first_model["obs_dates"], first_level["obs_native"], color="black", lw=2.2, label="observation", zorder=5)

        best_label = None
        best_rmse = np.inf
        for j, (label, model_item) in enumerate(variable["models"].items()):
            color = MODEL_COLORS[j % len(MODEL_COLORS)]
            d = model_item["levels"][level_pa]
            s = d["stats"]
            if np.isfinite(s["rmse"]) and s["rmse"] < best_rmse:
                best_rmse = s["rmse"]
                best_label = label
            axes[0].plot(model_item["model_dates"], d["model_at_level"], color=color, lw=1.9, label=label, zorder=2)
            axes[1].plot(model_item["model_dates"], d["diff"], color=color, lw=1.3, label=f"{label} - obs")

        axes[0].set_ylabel(f"{name} ({spec.units})")
        axes[0].legend(loc="best", frameon=False, ncols=2)
        axes[0].grid(True, alpha=0.25)

        axes[1].axhline(0, color="black", lw=0.8)
        axes[1].set_ylabel(f"diff ({spec.units})")
        axes[1].set_xlabel("Time")
        axes[1].legend(loc="best", frameon=False, ncols=2)
        axes[1].grid(True, alpha=0.25)

        axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4))
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        fig.suptitle(
            f"{name} at {level_pa / 100:.0f} hPa: multiple ARM97 model outputs vs observation\n"
            f"{spec.description} | best RMSE={best_rmse:.3g} ({best_label})",
            x=0.02,
            ha="left",
            y=0.98,
        )
        fig.autofmt_xdate(rotation=0)
        fig.subplots_adjust(top=0.86, left=0.08, right=0.98, bottom=0.10)
        display(fig)
        plt.close(fig)

    out = widgets.interactive_output(draw_profile, {"name": profile_var, "level_pa": pressure_level})
    display(widgets.VBox([profile_var, pressure_level]), out)
except Exception as exc:
    print("ipywidgets profile controls are unavailable in this kernel.")
    print(repr(exc))

## Save Tables

Run this cell after loading data if you want CSV summaries under `notebook_outputs/arm97_multiple_models_vs_observation`.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT_DIR / "surface_multi_model_summary.csv", index=False)
profile_summary.to_csv(OUT_DIR / "profile_multi_model_summary.csv", index=False)
print("wrote", OUT_DIR / "surface_multi_model_summary.csv")
print("wrote", OUT_DIR / "profile_multi_model_summary.csv")

## PDF Report

Run this cell after loading the surface and profile data to create a landscape PDF report.

Report structure:

- Page 1: summary table
- Surface section: one page per surface variable with model + observation and model - observation
- Profile section: one page per profile variable with selected-pressure-level time series

Optional `REPORT_START_TIME` and `REPORT_END_TIME` crop every report plot and recompute the report summary metrics for that time window without changing the selected variable or level count.


In [ ]:
from io import BytesIO
import textwrap

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


REPORT_PROFILE_VARS = None  # None includes all loaded PROFILE_DATA variables.
REPORT_PROFILE_MODELS = None  # None includes all configured models.
REPORT_PROFILE_LEVELS_HPA = [300, 500, 700, 850]

# Optional PDF-only time crop. Use strings like "1997-06-25" or "1997-06-25 12:00".
# Leave as None to keep the full model time range.
REPORT_START_TIME = None
REPORT_END_TIME = None

# REPORT_START_TIME = "1997-06-25"
# REPORT_END_TIME = "1997-07-05"


def _fmt_metric(value):
    if value is None or not np.isfinite(value):
        return "nan"
    return f"{value:.4g}"


def _model_report_rank(label):
    shown = display_label(label).lower()
    if "continuous" in shown:
        return 0
    if "stitched" in shown:
        return 1
    if "observation" in shown:
        return 9
    return 2


def _sorted_model_labels(items):
    labels = list(items)
    return sorted(labels, key=lambda label: (_model_report_rank(label), display_label(label).lower()))


def _time_window_label(start_time=None, end_time=None):
    start = "run start" if start_time is None else str(start_time)
    end = "run end" if end_time is None else str(end_time)
    return f"{start} to {end}"


def _time_window_mask(time_dates, start_time=None, end_time=None):
    stamps = pd.DatetimeIndex([pd.Timestamp(item) for item in time_dates])
    mask = np.ones(len(stamps), dtype=bool)
    if start_time is not None:
        mask &= stamps >= pd.Timestamp(start_time)
    if end_time is not None:
        mask &= stamps <= pd.Timestamp(end_time)
    if not mask.any():
        raise ValueError(f"time window has no samples: {_time_window_label(start_time, end_time)}")
    return mask


def _windowed_stats(model_dates, model_values, obs_values, start_time=None, end_time=None):
    mask = _time_window_mask(model_dates, start_time=start_time, end_time=end_time)
    return stats(np.asarray(model_values)[mask], np.asarray(obs_values)[mask])


def _apply_time_window(time_dates, *arrays, start_time=None, end_time=None):
    mask = _time_window_mask(time_dates, start_time=start_time, end_time=end_time)
    dates = np.asarray(time_dates, dtype=object)[mask]
    cropped = [np.asarray(arr)[mask] for arr in arrays]
    return (dates, *cropped)


def _available_profile_vars(requested=None):
    names = list(PROFILE_DATA)
    if requested is None:
        return names
    available = set(names)
    return [name for name in requested if name in available]


def _available_profile_models(variable_name, requested=None):
    labels = _sorted_model_labels(PROFILE_DATA[variable_name]["models"])
    if requested is None:
        return labels
    available = set(labels)
    return [label for label in requested if label in available]


def _surface_metric_table_rows(start_time=None, end_time=None):
    rows = []
    for variable, items in SURFACE_DATA.items():
        spec = next(iter(items.values()))["spec"]
        labels = _sorted_model_labels(items)
        row = [variable, spec.units]
        rmses = []
        n_value = ""
        for label in labels[:2]:
            d = items[label]
            s = _windowed_stats(
                d["model_dates"],
                d["model_values"],
                d["obs_at_model"],
                start_time=start_time,
                end_time=end_time,
            )
            rmse = s["rmse"]
            rmses.append(rmse)
            row.append(_fmt_metric(rmse))
            n_value = str(s["n"])
        while len(row) < 4:
            row.append("")
        if len(rmses) == 2 and np.isfinite(rmses[0]) and np.isfinite(rmses[1]):
            row.append(_fmt_metric(rmses[1] - rmses[0]))
        else:
            row.append("")
        row.append(n_value)
        rows.append(row)
    return rows


def _profile_mean_rmse(variable_name, model_label, start_time=None, end_time=None):
    item = PROFILE_DATA[variable_name]["models"][model_label]
    rmses = []
    for _, d in sorted(item["levels"].items()):
        s = _windowed_stats(
            item["model_dates"],
            d["model_at_level"],
            d["obs_at_model"],
            start_time=start_time,
            end_time=end_time,
        )
        rmses.append(s["rmse"])
    finite = np.asarray(rmses, dtype=float)
    finite = finite[np.isfinite(finite)]
    return float(np.nanmean(finite)) if finite.size else np.nan, len(item["levels"])


def _profile_summary_rows(profile_vars, profile_models, start_time=None, end_time=None):
    rows = []
    for variable in profile_vars:
        spec = PROFILE_DATA[variable]["spec"]
        labels = _available_profile_models(variable, profile_models)
        row = [variable, spec.units]
        means = []
        levels_count = ""
        for label in labels[:2]:
            mean_rmse, n_levels = _profile_mean_rmse(variable, label, start_time=start_time, end_time=end_time)
            means.append(mean_rmse)
            row.append(_fmt_metric(mean_rmse))
            levels_count = str(n_levels)
        while len(row) < 4:
            row.append("")
        if len(means) == 2 and np.isfinite(means[0]) and np.isfinite(means[1]):
            row.append(_fmt_metric(means[1] - means[0]))
        else:
            row.append("")
        row.append(levels_count)
        rows.append(row)
    return rows


def _fig_to_png(fig):
    image = BytesIO()
    fig.savefig(image, format="png", dpi=180, bbox_inches="tight")
    plt.close(fig)
    image.seek(0)
    return image


def _draw_surface_variable_figure(variable, items, start_time=None, end_time=None):
    spec = next(iter(items.values()))["spec"]
    labels = _sorted_model_labels(items)
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(11.0, 7.0),
        sharex=True,
        gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08},
    )

    first = next(iter(items.values()))
    obs_dates = np.asarray(first["obs_dates"], dtype=object)
    obs_native = np.asarray(first["obs_native"])
    if start_time is not None or end_time is not None:
        obs_mask = _time_window_mask(obs_dates, start_time=start_time, end_time=end_time)
        obs_dates = obs_dates[obs_mask]
        obs_native = obs_native[obs_mask]
    axes[0].plot(obs_dates, obs_native, color="#333333", lw=2.2, label="observation", zorder=5)

    for model_index, label in enumerate(labels):
        d = items[label]
        color = MODEL_COLORS[model_index % len(MODEL_COLORS)]
        shown_label = display_label(label)
        model_dates, model_values, diff = _apply_time_window(
            d["model_dates"],
            d["model_values"],
            d["diff"],
            start_time=start_time,
            end_time=end_time,
        )
        axes[0].plot(model_dates, model_values, color=color, lw=1.9, label=shown_label, zorder=3)
        axes[1].plot(model_dates, diff, color=color, lw=1.3, label=f"{shown_label} - obs")

    axes[0].set_ylabel(f"{variable} ({spec.units})")
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(loc="best", frameon=False, ncols=2)

    axes[1].axhline(0, color="#111111", lw=0.8)
    axes[1].set_ylabel(f"model - obs ({spec.units})")
    axes[1].set_xlabel("Time")
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(loc="best", frameon=False, ncols=2)
    axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

    rmse_text = []
    for label in labels:
        s = _windowed_stats(items[label]['model_dates'], items[label]['model_values'], items[label]['obs_at_model'], start_time=start_time, end_time=end_time)
        rmse_text.append(f"{display_label(label)} RMSE={s['rmse']:.3g}")
    fig.suptitle(
        f"{variable}: {spec.description}\nUnits: {spec.units}; " + ", ".join(rmse_text),
        x=0.02,
        y=0.985,
        ha="left",
        fontsize=13,
        fontweight="semibold",
    )
    fig.subplots_adjust(top=0.86, left=0.08, right=0.98, bottom=0.10)
    return _fig_to_png(fig)


def _nearest_profile_level_pa(variable_name, target_hpa):
    first_model = next(iter(PROFILE_DATA[variable_name]["models"].values()))
    levels_pa = np.array(sorted(first_model["levels"]), dtype=float)
    target_pa = float(target_hpa) * 100.0
    return float(levels_pa[np.nanargmin(np.abs(levels_pa - target_pa))])


def _selected_profile_levels_pa(variable_name, requested_hpa):
    selected = []
    seen = set()
    for target_hpa in requested_hpa:
        level_pa = _nearest_profile_level_pa(variable_name, target_hpa)
        if level_pa not in seen:
            selected.append(level_pa)
            seen.add(level_pa)
    return selected


def _draw_profile_selected_level_timeseries(variable_name, requested_levels_hpa, start_time=None, end_time=None):
    variable = PROFILE_DATA[variable_name]
    spec = variable["spec"]
    model_labels = _available_profile_models(variable_name, REPORT_PROFILE_MODELS)
    levels_pa = _selected_profile_levels_pa(variable_name, requested_levels_hpa)
    if not levels_pa:
        raise ValueError(f"No profile levels available for {variable_name}")

    n_levels = len(levels_pa)
    ncols = 2 if n_levels > 1 else 1
    nrows = int(np.ceil(n_levels / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(11.0, 7.0),
        sharex=True,
        squeeze=False,
        gridspec_kw={"hspace": 0.22, "wspace": 0.14},
    )
    axes_flat = axes.ravel()

    for panel_index, level_pa in enumerate(levels_pa):
        ax = axes_flat[panel_index]
        first_model = variable["models"][model_labels[0]]
        first_level = first_model["levels"][level_pa]
        obs_dates = np.asarray(first_model["obs_dates"], dtype=object)
        obs_native = np.asarray(first_level["obs_native"])
        if start_time is not None or end_time is not None:
            obs_mask = _time_window_mask(obs_dates, start_time=start_time, end_time=end_time)
            obs_dates = obs_dates[obs_mask]
            obs_native = obs_native[obs_mask]
        ax.plot(
            obs_dates,
            obs_native,
            color="#333333",
            lw=1.9,
            label="observation",
            zorder=5,
        )

        for model_index, model_label in enumerate(model_labels):
            model_item = variable["models"][model_label]
            level_item = model_item["levels"][level_pa]
            color = MODEL_COLORS[model_index % len(MODEL_COLORS)]
            shown_label = display_label(model_label)
            s = _windowed_stats(
                model_item["model_dates"],
                level_item["model_at_level"],
                level_item["obs_at_model"],
                start_time=start_time,
                end_time=end_time,
            )
            rmse = s["rmse"]
            model_dates, model_at_level = _apply_time_window(
                model_item["model_dates"],
                level_item["model_at_level"],
                start_time=start_time,
                end_time=end_time,
            )
            ax.plot(
                model_dates,
                model_at_level,
                color=color,
                lw=1.5,
                label=f"{shown_label} (RMSE={rmse:.3g})",
            )

        ax.set_title(f"{level_pa / 100.0:.0f} hPa", loc="left", fontsize=10, fontweight="semibold")
        ax.set_ylabel(spec.units)
        ax.grid(True, alpha=0.25)
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        if panel_index == 0:
            ax.legend(loc="best", frameon=False, fontsize=8)

    for ax in axes_flat[n_levels:]:
        ax.axis("off")
    for ax in axes[-1, :]:
        if ax.has_data():
            ax.set_xlabel("Time")

    fig.suptitle(
        f"{variable_name}: {spec.description}\nSelected pressure-level time series",
        x=0.02,
        y=0.985,
        ha="left",
        fontsize=13,
        fontweight="semibold",
    )
    fig.autofmt_xdate(rotation=0)
    fig.subplots_adjust(top=0.86, left=0.08, right=0.98, bottom=0.10)
    return _fig_to_png(fig)


def build_observation_pdf_report(output_path=None, profile_vars=REPORT_PROFILE_VARS, profile_models=REPORT_PROFILE_MODELS, start_time=REPORT_START_TIME, end_time=REPORT_END_TIME):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = Path(output_path or OUT_DIR / "ARM97_multiple_models_observation_report.pdf")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    profile_vars = _available_profile_vars(profile_vars)

    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=landscape(letter),
        rightMargin=0.35 * inch,
        leftMargin=0.35 * inch,
        topMargin=0.35 * inch,
        bottomMargin=0.35 * inch,
    )
    styles = getSampleStyleSheet()
    title_style = styles["Title"]
    title_style.fontSize = 18
    title_style.leading = 22
    body_style = styles["BodyText"]
    body_style.fontSize = 9
    body_style.leading = 11

    first_items = next(iter(SURFACE_DATA.values()))
    labels = _sorted_model_labels(first_items)
    display_labels = [display_label(label) for label in labels[:2]]
    while len(display_labels) < 2:
        display_labels.append("model")

    story = []
    story.append(Paragraph("ARM97 Multiple Model Outputs vs Observation", title_style))
    story.append(Spacer(1, 0.08 * inch))
    story.append(
        Paragraph(
            "Surface pages show model and observation time series plus model-minus-observation. Profile pages show selected-pressure-level model and observation time series.",
            body_style,
        )
    )
    story.append(Spacer(1, 0.06 * inch))
    story.append(Paragraph(f"Time window: {_time_window_label(start_time, end_time)}", body_style))
    story.append(Spacer(1, 0.12 * inch))


    report_headers = []
    for label in display_labels[:2]:
        lowered = label.lower()
        if "continuous" in lowered:
            report_headers.append("Continuous RMSE")
        elif "stitched" in lowered:
            report_headers.append("Stitched RMSE")
        else:
            report_headers.append(f"{label} RMSE")
    surface_table_data = [["Surface", "Units", report_headers[0], report_headers[1], "Delta", "n"]]
    surface_table_data.extend(_surface_metric_table_rows(start_time=start_time, end_time=end_time))
    surface_table = Table(surface_table_data, repeatRows=1, colWidths=[1.0 * inch, 0.75 * inch, 1.35 * inch, 1.35 * inch, 0.85 * inch, 0.65 * inch])
    surface_table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8EEF7")),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("FONTSIZE", (0, 0), (-1, -1), 7.2),
                ("ALIGN", (2, 1), (-1, -1), "RIGHT"),
                ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                ("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#CBD5E1")),
                ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F8FAFC")]),
            ]
        )
    )
    story.append(surface_table)
    story.append(Spacer(1, 0.12 * inch))

    profile_headers = []
    for label in display_labels[:2]:
        lowered = label.lower()
        if "continuous" in lowered:
            profile_headers.append("Continuous mean RMSE")
        elif "stitched" in lowered:
            profile_headers.append("Stitched mean RMSE")
        else:
            profile_headers.append(f"{label} mean RMSE")
    profile_table_data = [["Profile", "Units", profile_headers[0], profile_headers[1], "Delta", "Levels"]]
    profile_table_data.extend(_profile_summary_rows(profile_vars, profile_models, start_time=start_time, end_time=end_time))
    profile_table = Table(profile_table_data, repeatRows=1, colWidths=[0.9 * inch, 0.75 * inch, 1.45 * inch, 1.45 * inch, 0.85 * inch, 0.65 * inch])
    profile_table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8EEF7")),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("FONTSIZE", (0, 0), (-1, -1), 7.2),
                ("ALIGN", (3, 1), (-1, -1), "RIGHT"),
                ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                ("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#CBD5E1")),
                ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F8FAFC")]),
            ]
        )
    )
    story.append(profile_table)

    for variable, items in SURFACE_DATA.items():
        story.append(PageBreak())
        image = _draw_surface_variable_figure(variable, items, start_time=start_time, end_time=end_time)
        story.append(Image(image, width=10.2 * inch, height=6.45 * inch))

    for variable_name in profile_vars:
        story.append(PageBreak())
        image = _draw_profile_selected_level_timeseries(variable_name, REPORT_PROFILE_LEVELS_HPA, start_time=start_time, end_time=end_time)
        story.append(Image(image, width=10.2 * inch, height=6.45 * inch))

    doc.build(story)
    print(f"wrote {output_path}")
    return output_path


REPORT_PDF = build_observation_pdf_report()
REPORT_PDF
